## <img style="float: left; padding-right: 10px; width: 100px" src="https://raw.githubusercontent.com/sbildea/SEED/main/imagini/unitbv_mie.png"> <center>Știința Explorării și Exploatării Datelor - SEED1</center>


##    <center>Extra 3: Asociere și recomandări (cod complet)</center>
**Universitatea Transilvania**<br>
**SIIPA, Semestrul I**<br>
**Instructor:** Conf. dr. Teodor &Scedil;tefan B&icirc;ldea<br>
**Contact:** teodor.bildea@unitbv.ro<br>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sbildea/SEED/blob/main/cursuri/03-invatare-nesupervizata/extra-03-asociere-recomandari.ipynb)

In [ ]:
# Configurare curs SEED: rulați această celulă prima.
# Colab: descarcă materialele cursului și instalează pachetele lipsă (aprox. un minut).
# Local: găsește singură rădăcina repository-ului, oricare ar fi folderul curent al kernelului.
import os, sys
MODUL = "03-invatare-nesupervizata"
if "google.colab" in sys.modules:
    if not os.path.exists("/content/SEED"):
        !git clone -q --depth 1 https://github.com/sbildea/SEED.git /content/SEED
        !pip install -q -r /content/SEED/requirements.txt
    RADACINA = "/content/SEED"
else:
    RADACINA = os.path.abspath(".")
    while not os.path.isdir(os.path.join(RADACINA, "seed")) and os.path.dirname(RADACINA) != RADACINA:
        RADACINA = os.path.dirname(RADACINA)
    import importlib.util, socket
    lipsa = [p for p in ("pandas", "matplotlib", "seaborn", "sklearn") if importlib.util.find_spec(p) is None]
    if lipsa:
        print("Lipsesc pachetele", lipsa, "- în terminal: pip install -r requirements-local.txt")
    try:
        socket.create_connection(("raw.githubusercontent.com", 443), 2).close()
    except OSError:
        print("Fără internet: imaginile din text nu se văd. O singură dată, în terminal: python tools/imagini_offline.py")
os.chdir(os.path.join(RADACINA, "cursuri", MODUL))
sys.path.insert(0, RADACINA)

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

# Set de date - MovieLens data setul mic

In [1]:
import os, requests, zipfile, io
# datele MovieLens sunt deja în folderul modulului; se descarcă doar dacă lipsesc
if not os.path.exists('ml-latest-small'):
    zipfile.ZipFile(io.BytesIO(requests.get('https://files.grouplens.org/datasets/movielens/ml-latest-small.zip').content)).extractall()

In [3]:
data_movies = pd.read_csv('ml-latest-small/movies.csv')
data_tags = pd.read_csv('ml-latest-small/tags.csv')
data_ratings = pd.read_csv('ml-latest-small/ratings.csv')

### Date despre filme
Acesta este singurul set de date care conține titlul filmelor și cheia unică a fiecărui film

In [4]:
data_movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [18]:
data_movies['year'] = data_movies['title'].apply(lambda x: int(x.strip().split('(')[-1][:-1]) if x.strip()[-1]==')' and x.strip()[-6]=='(' and x.strip()[-5:-1].isnumeric() else 0)
data_movies.year.max()



In [17]:
x = "(1995)"
print( x.strip()[-6])
x.split('(')[-1][:-1] if x.strip()[-1]==')' and x.strip()[-6]=='(' and x.strip()[-5:-1].isnumeric() else 0

(


'1995'

In [9]:
data_movies[data_movies.year =='The O']

,movieId,title,genres,year
9448,167570,The OA,(no genres listed),The O


In [19]:
data_movies.movieId.nunique(),print('length of the dataset: ',len(data_movies))

length of the dataset:  9742


(9742, None)

### Etichete despre filme (tags)
Acest set de date conține etichetele adăugate filmului plus ID-ul de utilizator, care este valoros pentru a fi considerat ca un index pentru fiecare tranzacție, pe de altă parte, setul de date are mai puține ID-uri de film decât setul de date de evaluare ceea ce m-a făcut să-l selectez.

In [20]:
data_tags

,userId,movieId,tag,timestamp
0,2,60756,funny,1445714994
1,2,60756,Highly quotable,1445714996
2,2,60756,will ferrell,1445714992
3,2,89774,Boxing story,1445715207
4,2,89774,MMA,1445715200
...,...,...,...,...
3678,606,7382,for katie,1171234019
3679,606,7936,austere,1173392334
3680,610,3265,gun fu,1493843984
3681,610,3265,heroic bloodshed,1493843978


In [21]:
data_tags.movieId.nunique()

1572

In [22]:
data_tags.tag.value_counts().sort_values(ascending=False)

tag
In Netflix queue     131
atmospheric           36
thought-provoking     24
superhero             24
funny                 23
                    ... 
storytelling           1
travolta               1
Uma Thurman            1
unusual                1
Heroic Bloodshed       1
Name: count, Length: 1589, dtype: int64

### Date de evaluare a filmelor (ratinguri)
Datele de evaluare a filmelor conțin mai mult de 9k ID-uri de filme, care pot fi utile pentru a fi combinate cu setul de date pentru titlurile filmelor.

In [15]:
data_ratings

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931
...,...,...,...,...
100831,610,166534,4.0,1493848402
100832,610,168248,5.0,1493850091
100833,610,168250,5.0,1494273047
100834,610,168252,5.0,1493846352


In [23]:
pd.DataFrame(data_ratings.rating.value_counts()).reset_index().sort_values(by='rating')

,rating,count
9,0.5,1370
7,1.0,2811
8,1.5,1791
5,2.0,7551
6,2.5,5550
1,3.0,20047
3,3.5,13136
0,4.0,26818
4,4.5,8551
2,5.0,13211


In [16]:
data_ratings.movieId.nunique()

9724

Combinăm seturile de date pentru a crea obiectul de studiu, obiectul muncii în orice experiment: the big fat data science table!

In [24]:
df_movies_full = data_movies.merge(
    data_tags,
    on = 'movieId',
    how = 'inner')

In [22]:
df_movies_full.nunique()

movieId      1572
title        1572
genres        372
year           99
userId         58
tag          1589
timestamp    3411
dtype: int64

In [ ]:
df_movies_full

# APRIORI

Ne trebui un fel de coș de cumpărături, deci lista filmelor fiecărui utilizator, fără să ne intereseze nimic despre utilizator (ratinguri acordate de exemplu...).

Pentru început aplicăm One-Hot-Encoding pe filme. Astfel filmele vor deveni coloane iar o linie din vechiul tabel cu userId, title va deveni userdId, urmat de un șir de 1572 de valori: toate zero cu excepția celei care corespunde coloanei cu numele title.

Observați că avem acelați număr de linii ca înainte dar numărul de coloane e numărul filmelor + 1.

In [25]:
df_movies_full_OHE =pd.get_dummies( df_movies_full[['userId','title']],columns = [ 'title'])
df_movies_full_OHE.shape

(3683, 1573)

In [25]:
df_movies_full_OHE.userId.nunique()

58

Acum comasăm informația deoarece ne interesează o lista cu filmele văzute de un utilizator. 

Pentru asta vom face un groupby pe utilizatori și vom agrega cu funcția max valorile fiecărui film. Astfel la final vom obține o lista cu 1 pentru filmele văzute de utilizator, restul 0. Exact ce ne trebuie pentru Apriori.

In [26]:
df_user_movies = df_movies_full_OHE.drop_duplicates().groupby(by = ["userId"], as_index = False).max()
df_user_movies.head()

,userId,title_(500) Days of Summer (2009),title_...And Justice for All (1979),title_10 Cloverfield Lane (2016),title_10 Things I Hate About You (1999),title_101 Dalmatians (1996),title_101 Dalmatians (One Hundred and One Dalmatians) (1961),"title_11'09""01 - September 11 (2002)",title_12 Angry Men (1957),title_127 Hours (2010),title_13 Going on 30 (2004),title_2001: A Space Odyssey (1968),title_21 Grams (2003),title_25th Hour (2002),title_28 Days Later (2002),"title_39 Steps, The (1935)",title_3:10 to Yuma (2007),"title_40-Year-Old Virgin, The (2005)","title_400 Blows, The (Les quatre cents coups) (1959)",title_42 Up (1998),title_84 Charing Cross Road (1987),title_8MM (1999),title_A Million Ways to Die in the West (2014),title_A Pigeon Sat on a Branch Reflecting on Existence (2014),title_A Story of Children and Film (2013),title_A.I. Artificial Intelligence (2001),title_About a Boy (2002),"title_Accused, The (1988)",title_Adam's Rib (1949),title_Addams Family Values (1993),"title_Addams Family, The (1991)","title_Adventures of Priscilla, Queen of the Desert, The (1994)","title_Adventures of Robin Hood, The (1938)","title_African Queen, The (1951)",title_After the Thin Man (1936),"title_Age of Innocence, The (1993)",title_Air Force One (1997),title_Airheads (1994),title_Airplane! (1980),title_Akira (1988),title_Aladdin (1992),title_Alfie (1966),title_Alice Adams (1935),title_Alice Doesn't Live Here Anymore (1974),title_Alice in Wonderland (1951),title_Alien (1979),title_Aliens (1986),title_Alive (1993),title_All About Eve (1950),title_All the King's Men (1949),title_All the President's Men (1976),title_All the Real Girls (2003),title_Almost Famous (2000),title_Amadeus (1984),"title_Amelie (Fabuleux destin d'Amélie Poulain, Le) (2001)",title_Amen. (2002),title_America's Sweethearts (2001),title_American Gangster (2007),title_American History X (1998),title_American Movie (1999),title_American Pie (1999),"title_American President, The (1995)",title_American Splendor (2003),"title_American in Paris, An (1951)","title_Americanization of Emily, The (1964)",title_Anastasia (1956),title_Anatomy of a Murder (1959),title_Anchorman 2: The Legend Continues (2013),title_Anchorman: The Legend of Ron Burgundy (2004),title_And Then There Were None (1945),title_And the Band Played On (1993),"title_Andalusian Dog, An (Chien andalou, Un) (1929)",title_Angel's Egg (Tenshi no tamago) (1985),title_Angie (1994),title_Animal House (1978),"title_Animatrix, The (2003)",title_Anna Karenina (1997),title_Anne Frank Remembered (1995),title_Anne of the Thousand Days (1969),title_Annie Hall (1977),title_Another Thin Man (1939),"title_Apartment, The (1960)",title_Apocalypse Now (1979),title_Apollo 13 (1995),title_Arachnophobia (1990),title_Argo (2012),"title_Aristocats, The (1970)",title_Armageddon (1998),title_Around the World in 80 Days (1956),title_Arrival (2016),"title_Arrival, The (1996)",title_Arsenic and Old Lace (1944),"title_Astronaut's Wife, The (1999)",title_Au Hasard Balthazar (1966),title_Auntie Mame (1958),title_Avalon (1990),title_Avatar (2009),"title_Avengers, The (2012)",title_Avengers: Infinity War - Part I (2018),"title_Aviator, The (2004)","title_Awful Truth, The (1937)","title_Babadook, The (2014)",title_Babe (1995),title_Babel (2006),title_Babette's Feast (Babettes gæstebud) (1987),title_Babylon 5: In the Beginning (1998),"title_Bachelor and the Bobby-Soxer, The (1947)",title_Back to the Future (1985),title_Back to the Future Part II (1989),title_Bad Day at Black Rock (1955),"title_Bad News Bears, The (1976)","title_Bad Seed, The (1956)","title_Bad and the Beautiful, The (1952)","title_Ballad of Jack and Rose, The (2005)","title_Bank Job, The (2008)",title_Barton Fink (1991),"title_Basketball Diaries, The (1995)",title_Batman (1989),title_Batman Forever (1995),title_Batman Returns (1992),title_Batman v Superman: Dawn of Justice (2016),title_Battle Royale (Batoru rowaiaru) (2000),"title_Battle of Algiers, The (La battaglia di Algeri) (196

Să ne verificăm un pic:

In [27]:
print(df_user_movies.userId.nunique(),df_movies_full.userId.nunique())

58 58


„Coșul de cumpărături” are 58 de tranzacții - liste de filme văzute de utilizatori.

In [31]:
df= df_user_movies.drop(columns = 'userId')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 58 entries, 0 to 57
Columns: 1572 entries, title_(500) Days of Summer (2009) to title_eXistenZ (1999)
dtypes: bool(1572)
memory usage: 89.2 KB


Pentru a nu complica prea tare calculele generăm doar 1-itemseturi și 2-itemseturi cu support minim = 0.01 (avem 58 de tranzacții, pentru 1-itemseturi asta înseamnă 1 tranzacție).

In [29]:
! pip install mlxtend

In [32]:
from mlxtend.frequent_patterns import apriori
%time

apriori_frequent_itemsets = apriori(df, 
                                    min_support=0.01, # suport minim căutat
                                    use_colnames=True,
                                    max_len=2 # max K
                                   )
apriori_frequent_itemsets.head()

CPU times: total: 0 ns
Wall time: 0 ns


,support,itemsets
0,0.017241,(title_(500) Days of Summer (2009))
1,0.017241,(title_...And Justice for All (1979))
2,0.017241,(title_10 Cloverfield Lane (2016))
3,0.017241,(title_10 Things I Hate About You (1999))
4,0.017241,(title_101 Dalmatians (1996))


In [33]:
apriori_frequent_itemsets['itemsets'].apply(lambda x: len(x)).value_counts()

itemsets
2    774986
1      1572
Name: count, dtype: int64

Ștampilăm lungimea pe fiecare înregistrare.

In [34]:
apriori_frequent_itemsets['length'] = apriori_frequent_itemsets['itemsets'].apply(lambda x: len(x))
apriori_frequent_itemsets.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 776558 entries, 0 to 776557
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   support   776558 non-null  float64
 1   itemsets  776558 non-null  object 
 2   length    776558 non-null  int64  
dtypes: float64(1), int64(1), object(1)
memory usage: 17.8+ MB


În sfârșit putem chema generatorul de reguli de asociere:

In [35]:
%%time
from mlxtend.frequent_patterns import association_rules
sample_apriori_res = association_rules(apriori_frequent_itemsets,min_threshold=0.01)

CPU times: total: 6.03 s
Wall time: 6.05 s


In [36]:
sample_apriori_res.columns

Index(['antecedents', 'consequents', 'antecedent support', 'consequent support', 'support', 'confidence', 'lift', 'representativity', 'leverage', 'conviction', 'zhangs_metric', 'jaccard', 'certainty', 'kulczynski'], dtype='object')

In [37]:
sample_apriori_res.sort_values(by = ['lift'], ascending = [False])

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,(title_Akira (1988)),(title_(500) Days of Summer (2009)),0.017241,0.017241,0.017241,1.00,58.00,1.0,0.016944,inf,1.000000,1.000000,1.000000,1.000
976218,"(title_House of Mirth, The (2000))",(title_Mildred Pierce (1945)),0.017241,0.017241,0.017241,1.00,58.00,1.0,0.016944,inf,1.000000,1.000000,1.000000,1.000
976226,"(title_House of Mirth, The (2000))",(title_Millions (2004)),0.017241,0.017241,0.017241,1.00,58.00,1.0,0.016944,inf,1.000000,1.000000,1.000000,1.000
976225,"(title_House of Mirth, The (2000))",(title_Million Dollar Baby (2004)),0.017241,0.017241,0.017241,1.00,58.00,1.0,0.016944,inf,1.000000,1.000000,1.000000,1.000
976224,(title_Million Dollar Baby (2004)),"(title_House of Mirth, The (2000))",0.017241,0.017241,0.017241,1.00,58.00,1.0,0.016944,inf,1.000000,1.000000,1.000000,1.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
118901,(title_Anchorman: The Legend of Ron Burgundy (...,(title_Star Wars: Episode IV - A New Hope (1977)),0.068966,0.172414,0.017241,0.25,1.45,1.0,0.005351,1.103448,0.333333,0.076923,0.093750,0.175
118900,(title_Star Wars: Episode IV - A New Hope (1977)),(title_Anchorman: The Legend of Ron Burgundy (...,0.172414,0.068966,0.017241,0.10,1.45,1.0,0.005351,1.034483,0.375000,0.076923,0.033333,0.175
1029677,(title_Star Wars: Episode IV - A New Hope (1977)),(title_Inception (2010)),0.172414,0.068966,0.017241,0.10,1.45,1.0,0.005351,1.034483,0.375000,0.076923,0.033333,0.175
647031,(title_Dr. Strangelove or: How I Learned to St...,(title_Star Wars: Episode IV - A New Hope (1977)),0.086207,0.172414,0.017241,0.20,1.16,1.0,0.002378,1.034483,0.150943,0.071429,0.033333,0.150


`Discuție` Puteți încerca cu 3-itemseturi. Veți obține mai multe reguli, printre care două titluri care, văzute împreună, recomandă un al treilea.

# Factorizarea matricei utilizator × film (factori latenți)

În anii trecuți această secțiune folosea pachetul `lightfm`, care nu mai este întreținut și nu se mai instalează pe versiunile noi de Python. Ideea de bază este aceeași și o putem arăta doar cu `scikit-learn`: **factorizarea matricei** utilizator × film.

Fiecare utilizator și fiecare film primesc câte un vector scurt de **factori latenți** (de exemplu 30 de numere). Un factor poate corespunde, aproximativ, unui gen sau unui stil („acțiune", „film de autor"), dar nu îi dăm noi sensul: îl învață modelul din date. Scorul unei perechi (utilizator, film) este produsul scalar al celor doi vectori; filmele cu scor mare, încă nevăzute, devin recomandări.

## Datele: feedback implicit

Folosim din nou MovieLens (`data_ratings`, `data_movies`, încărcate mai sus). Ca în exemplul original, nu prezicem nota, ci doar **dacă** utilizatorului i-a plăcut filmul: considerăm pozitive notele de cel puțin 4. Împărțim interacțiunile aleator în antrenare (80%) și testare (20%).

In [ ]:
import numpy as np
from scipy import sparse
from sklearn.model_selection import train_test_split

pozitive = data_ratings[data_ratings.rating >= 4][["userId", "movieId"]]

# codificăm utilizatorii și filmele ca indici 0..n-1
utilizatori = pozitive.userId.astype("category")
filme = pozitive.movieId.astype("category")
pozitive = pozitive.assign(u=utilizatori.cat.codes, i=filme.cat.codes)
id_filme = filme.cat.categories          # indice -> movieId
n_u, n_i = int(pozitive.u.max()) + 1, int(pozitive.i.max()) + 1

antr, test = train_test_split(pozitive, test_size=0.2, random_state=17)

def matrice(df):
    return sparse.csr_matrix((np.ones(len(df)), (df.u, df.i)), shape=(n_u, n_i))

M_antr, M_test = matrice(antr), matrice(test)
print(f"{n_u} utilizatori, {n_i} filme; {M_antr.nnz} interacțiuni de antrenare, {M_test.nnz} de testare")
print(f"densitate: {M_antr.nnz / (n_u * n_i):.2%} din matrice este completată")

## Modelul

`TruncatedSVD` descompune matricea rară în factori latenți: `U` (utilizatori × k) și `V` (k × filme). Produsul lor reconstruiește o matrice **densă** de scoruri, cu valori și pentru filmele pe care utilizatorul nu le-a văzut încă.

In [ ]:
from sklearn.decomposition import TruncatedSVD

svd = TruncatedSVD(n_components=30, random_state=17)
U = svd.fit_transform(M_antr)      # utilizatori × 30
V = svd.components_                # 30 × filme
scoruri = U @ V
print(U.shape, V.shape, scoruri.shape)

## Evaluarea recomandărilor: precision@k

Pentru fiecare utilizator recomandăm primele *k* filme pe care **nu** le-a văzut la antrenare și ne uităm câte dintre ele apar în setul lui de testare. Aceasta este **precision@k**.

Un număr singur nu spune mult, așa că îl comparăm cu o referință simplă: recomandăm tuturor **cele mai populare** filme. Un model care nu bate popularitatea nu merită folosit.

In [ ]:
def precizie_la_k(scoruri, k):
    s = scoruri.copy()
    s[M_antr.nonzero()] = -np.inf                 # nu recomandăm ce a văzut deja
    top = np.argsort(-s, axis=1)[:, :k]
    relevante = M_test.toarray()
    loviri = np.take_along_axis(relevante, top, axis=1).sum(axis=1)
    au_test = relevante.sum(axis=1) > 0           # doar utilizatorii cu filme în test
    return (loviri[au_test] / k).mean()

popularitate = np.tile(np.asarray(M_antr.sum(axis=0)), (n_u, 1)).astype(float)

for k in [5, 10, 20]:
    print(f"k = {k:2d}:  SVD {precizie_la_k(scoruri, k):.3f}   popularitate {precizie_la_k(popularitate, k):.3f}")

Valorile par mici, dar fiecare utilizator are doar câteva filme în test din peste 6.000 posibile, deci o recomandare nimerită la întâmplare ar fi extrem de rară. Comparația cu popularitatea este cea care contează.

**De încercat:** schimbați `n_components` (5, 30, 100). Cu prea puțini factori modelul seamănă cu popularitatea; cu prea mulți începe să memoreze datele de antrenare (supraînvățare), același compromis ca în modulul de regresie.

## Recomandări pentru câțiva utilizatori

Pentru o judecată calitativă, afișăm câteva filme apreciate de un utilizator și primele recomandări.

In [ ]:
titlu = data_movies.set_index("movieId").title

def recomandari(u, n=5):
    vazute = M_antr[u].indices
    s = scoruri[u].copy()
    s[vazute] = -np.inf
    print(f"Utilizatorul {utilizatori.cat.categories[u]}")
    print("   I-au plăcut:", "; ".join(titlu[id_filme[vazute[:3]]]))
    print("   Recomandări:", "; ".join(titlu[id_filme[np.argsort(-s)[:n]]]))

for u in [3, 25, 450]:
    recomandari(u)

Subiectul este vast. Voi încheia cu menționarea unui studiu de caz întâlnit de mine. Este necesar să fi auzit de acești algoritmi pentru a înțelege cum puteți formula diverse probleme în probleme de recomandare care să fie rezolvate prin factorizarea matricei.

<b>Problemă</b> Un mare distribuitor de piese auto (70+ depozite în România, plus câteva în afara țării) m-a abordat pentru un algoritm de extindere a gamei oferite de depozitele locale. Din motive de logistică și de estimare a cererii și ofertei, nu toate piesele sunt stocate la toate depozitele. Gama pieselor stocate (și volumul lor) este o problemă de maximă importanță.

<b>Soluție</b>
Am formulat problema ca o problemă de utilizator/ratinguri unde utilizatorii sunt depozitele iar ratingurl l-am inventat pornind de la vănzările unei piese în depozit. Vă aduceți aminte tema de discretizare cu coloana de referință? Un rating de la 1 la 5 poate fi obținut organizănd produsele după venitul sau profitul adus. Astfel am putut genera recomandări pornind de la vănzarile zilnice, pentru fiecare depozit.

<b>Rezultat</b> Am generat lista de recomandări, și codul care poate genera o astfel de listă la fiecare reantrenare, pentru a putea oferi candidați noi, neîncercați, la fiecare extindere de gamă efectuată de companie.